## Sample OSM Polygons and Get (Uncropped) EO Images

### This notebook samples building polygons from a predefined list of cities using OpenStreetMap (OSM) via the osmnx library. It extracts the center coordinates of a random subset of these buildings and automates the downloading of satellite imagery using the Google Maps Static API, utilizing robust retry and logging logic.

In [ ]:
import os
import time
import requests
import pandas as pd
import osmnx as ox
import warnings

# Suppress GeoPandas UserWarning regarding centroid calculation on geographic CRS
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

# === CONFIGURATION AND API SETUP ===

# Load Google Maps Static API key
API_KEY = "" # <-- Insert your Google Maps Static API key here

# Output directory where the downloaded satellite images will be stored
OUT_DIR = "osm_sampled_roofs" 

# CSV file to record metadata for any downloads that fail after multiple attempts
FAILED_CSV = "failed_downloads.csv" 

# List of cities to sample from (ensure the naming is clear for OSM Nominatim geocoder)
CITIES = [
    "New York, United States",
    "Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, India",
    "Dar es-Salaam, Coastal Zone, Tanzania"
]

# Number of building polygons to randomly sample per city to avoid exhausting API limits
SAMPLES_PER_CITY = 20

# Ensure the output directory exists on the filesystem
os.makedirs(OUT_DIR, exist_ok=True) 

# Initialize a persistent requests session to improve performance over multiple requests
session = requests.Session()

In [ ]:
# === HELPER FUNCTIONS ===

def sample_buildings_from_cities(cities, n_samples):
    """
    Fetches building footprints from OSM for a list of cities, samples them, 
    and extracts their centroid coordinates.
    """
    building_records = []
    
    for city in cities:
        print(f"Fetching building footprints for: {city}...")
        try:
            # Fetch building features from OSM
            gdf = ox.features_from_place(city, tags={'building': True})
            
            # Filter for Polygons and MultiPolygons to ensure we have valid building footprints
            gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]
            
            # Sample a subset to avoid massive API bills
            if len(gdf) > n_samples:
                gdf = gdf.sample(n=n_samples, random_state=42)
            
            # Extract centroids (lat/lon) for the Google Static Maps API center point
            for idx, row in gdf.iterrows():
                centroid = row.geometry.centroid
                
                # Create a clean prefix for the filename based on the city name
                city_prefix = city.split(',')[0].replace(' ', '_').lower()
                
                # Handle potential MultiIndex from osmnx (element_type, osmid)
                osmid = idx[1] if isinstance(idx, tuple) else idx
                
                building_records.append({
                    "filename": f"{city_prefix}_{osmid}.png",
                    "latitude": centroid.y,
                    "longitude": centroid.x,
                    "city": city
                })
            print(f"Successfully sampled {len(gdf)} buildings from {city}.")
            
        except Exception as e:
            print(f"Error fetching OSM data for {city}: {e}")
            
    return pd.DataFrame(building_records)

def get_image(lat, lon):
    """
    Constructs and sends a request to the Google Static Maps API for a satellite 
    image centered at the provided coordinates.
    """
    url = (
        "https://maps.googleapis.com/maps/api/staticmap"
        f"?center={lat},{lon}"
        "&zoom=20"            # High-resolution building views
        "&size=256x256"       # Request 256x256 pixel tiles
        "&maptype=satellite"  # Use satellite imagery layer
        f"&key={API_KEY}"
    )
    return session.get(url, timeout=30)

In [ ]:
# === STEP 1: GENERATE DOWNLOAD QUEUE FROM OSM ===

print("=== Starting OSM Building Sampling ===")
df = sample_buildings_from_cities(CITIES, SAMPLES_PER_CITY)

if df.empty:
    print("No buildings successfully sampled. Exiting.")
    exit()

print(f"\nTotal buildings to process: {len(df)}")

# === STEP 2: MAIN DOWNLOAD LOOP ===

print("\n=== Starting Google Static Maps Downloads ===")
failed_rows = [] 

for i, row in df.iterrows():
    filename = row["filename"]
    lat = row["latitude"]
    lon = row["longitude"]

    # Define the full destination path for the current image
    out_path = os.path.join(OUT_DIR, filename) 

    # Skip download if the file already exists locally
    if os.path.exists(out_path):
        continue

    # Skip download if coordinates are missing or invalid
    if pd.isna(lat) or pd.isna(lon): 
        print(f"Bad coordinates: {filename} | lat={lat} lon={lon}")
        failed_rows.append({
            "filename": filename,
            "latitude": lat,
            "longitude": lon,
            "reason": "bad_coordinates"
        })
        continue

    success = False 

    # Attempt to download the image up to 5 times
    for attempt in range(1, 6): 
        try:
            response = get_image(lat, lon)

            content_type = response.headers.get("Content-Type", "")
            body_preview = response.text[:200] if "text" in content_type.lower() else ""

            # Check for HTTP 200 (OK) and a valid image MIME type
            if response.status_code == 200 and "image" in content_type.lower(): 
                with open(out_path, "wb") as f:
                    f.write(response.content)
                success = True
                break
            
            print(
                f"Attempt {attempt}/5 failed for {filename} | "
                f"status={response.status_code} content_type={content_type} "
                f"body={body_preview}"
            )

        except Exception as e:
            print(f"Attempt {attempt}/5 error for {filename}: {e}")

        # Exponential backoff
        time.sleep(2 * attempt) 

    # If all 5 attempts fail, log it
    if not success:
        failed_rows.append({
            "filename": filename,
            "latitude": lat,
            "longitude": lon,
            "reason": "request_failed"
        })

    if (i + 1) % 10 == 0: 
        print(f"Processed {i + 1}/{len(df)}")

    # Short delay to prevent rate limiting
    time.sleep(0.2)

# === FINALIZATION ===

if failed_rows: 
    pd.DataFrame(failed_rows).to_csv(FAILED_CSV, index=False)
    print(f"\nCompleted with some errors. Saved failed rows to {FAILED_CSV}")
else:
    print("\nCompleted successfully. No failed rows.")